# MMAudio ile Ses Üretimi

**MMAudio** video ve/veya metin girdisinden ses üreten bir yapay zeka modelidir.

Bu notebook'ta üç mod desteklenir:
- **Video + Text → Audio** — Bir video yükleyip metin ile yönlendirin (en iyi kalite)
- **Video → Audio** — Sadece video verin, model sesi otomatik üretsin
- **Text → Audio** — Sadece metin açıklaması ile ses üretin

Tüm parametreler **Konfigürasyon** hücresinde toplanmıştır — farklı değerlerle deney yapabilirsiniz.

**Model:** `large_44k_v2` (44kHz, yüksek kalite) veya `small_16k` (16kHz, hızlı)  
**Lisans:** Model ağırlıkları CC-BY-NC 4.0 (yalnızca ticari olmayan kullanım)  
**GitHub:** https://github.com/hkchengrex/MMAudio

## GPU Ayarı

Bu notebook GPU gerektirir. T4 GPU (ücretsiz) yeterlidir.

1. Üst menüden **Runtime** → **Change runtime type** tıklayın
2. **Hardware accelerator** altından **T4 GPU** seçin
3. **Save** tıklayın

Aşağıdaki hücreyi çalıştırarak GPU'nun bağlı olduğunu doğrulayın.

In [ ]:
# GPU bagli mi kontrol et
import torch

if torch.cuda.is_available():
    # GPU adini ve bellek bilgisini yazdir
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_memory = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / (1024**3)
    print(f"GPU bulundu: {gpu_name}")
    print(f"GPU bellegi: {gpu_memory:.1f} GB")
else:
    # GPU yoksa uyari ver
    print("UYARI: GPU bulunamadi!")
    print("Lutfen Runtime > Change runtime type > T4 GPU secin.")
    print("GPU olmadan model cok yavas calisir.")

## Konfigürasyon

Aşağıdaki hücrede tüm ayarlanabilir parametreler bir arada verilmiştir.  
Değerleri değiştirip tekrar çalıştırarak farklı sonuçları deneyebilirsiniz.

### Parametre Açıklamaları

| Parametre | Default | Açıklama |
|-----------|---------|----------|
| **PROMPT** | — | Üretmek istediğiniz sesi tarif edin (İngilizce) |
| **NEGATIVE_PROMPT** | `""` | İstemediğiniz sesleri tarif edin (ör. `"music, speech"`) |
| **DURATION** | `8` | Ses süresi (saniye). Model 8s ile eğitildi, farklı değerler kaliteyi düşürebilir |
| **VIDEO_PATH** | `None` | Video dosyası yolu. Verilirse video-to-audio modu aktif olur |
| **MODEL_NAME** | `"large_44k_v2"` | Model seçimi: `"large_44k_v2"` (44kHz, kaliteli) veya `"small_16k"` (16kHz, hızlı) |
| **SEED** | `42` | Rastgele sayı tohumu. Aynı seed = aynı sonuç (tekrarlanabilirlik) |
| **NUM_STEPS** | `25` | Flow matching adım sayısı. Fazla = kaliteli ama yavaş, az = hızlı ama kaba |
| **INFERENCE_MODE** | `"euler"` | `"euler"` (sabit adım, hızlı) veya `"adaptive"` (otomatik adım, daha doğru) |
| **CFG_STRENGTH** | `4.5` | Rehberlik gücü. Yüksek = prompt'a daha sadık, düşük = daha serbest |

**Örnek prompt'lar:**
- `"rain falling on a rooftop at night"`
- `"a dog barking in a park"`
- `"ocean waves crashing on rocks"`
- `"thunder and lightning during a storm"`

In [ ]:
# ============================================================
#  KONFIGURASYON - Tum ayarlari buradan degistirin
# ============================================================

# --- Giris ---
# Uretmek istediginiz sesi tarif edin (Ingilizce)
PROMPT = "rain falling on a rooftop at night"

# Istemediginiz sesleri tarif edin (bos birakabilirsiniz)
# Ornek: "music, speech, noise"
NEGATIVE_PROMPT = ""

# Video dosyasi yolu (None = sadece text-to-audio)
# Video yuklerseniz video-to-audio moduna gecer
# Ornek: "/content/my_video.mp4"
VIDEO_PATH = None

# --- Uretim Ayarlari ---
# Ses suresi (saniye) - model 8s ile egitildi, farkli degerler kaliteyi dusurebilir
DURATION = 8

# Rastgele sayi tohumu - ayni seed ayni sonucu verir (tekrarlanabilirlik)
SEED = 42

# --- Model ---
# "large_44k_v2" = 44kHz, yuksek kalite (onerilen)
# "small_16k"    = 16kHz, daha hizli ama dusuk kalite
MODEL_NAME = "large_44k_v2"

# --- Flow Matching ---
# Adim sayisi: fazla = kaliteli ama yavas, az = hizli ama kaba
# Onerilen aralik: 10-50, default: 25
NUM_STEPS = 25

# Cozucu modu:
#   "euler"    = sabit adim, hizli ve tahmin edilebilir
#   "adaptive" = otomatik adim boyutu, daha dogru (NUM_STEPS gormezden gelinir)
INFERENCE_MODE = "euler"

# --- Rehberlik ---
# Classifier-free guidance gucu
# Yuksek = prompt'a daha sadik, dusuk = daha serbest
# Onerilen aralik: 1.0-10.0, default: 4.5
CFG_STRENGTH = 4.5

# ============================================================
print("=== Konfigurasyon ===")
print(f"  Prompt:          {PROMPT}")
print(f"  Negatif Prompt:  {NEGATIVE_PROMPT or '(yok)'}")
print(f"  Video:           {VIDEO_PATH or '(yok - text-only mod)'}")
print(f"  Sure:            {DURATION} saniye")
print(f"  Seed:            {SEED}")
print(f"  Model:           {MODEL_NAME}")
print(f"  Adim Sayisi:     {NUM_STEPS}")
print(f"  Cozucu:          {INFERENCE_MODE}")
print(f"  CFG Gucu:        {CFG_STRENGTH}")
print("=" * 25)

## Kurulum

Şimdi MMAudio reposunu klonlayıp gerekli kütüphaneleri yükleyeceğiz.  
Bu adım birkaç dakika sürebilir.

In [ ]:
# MMAudio reposunu klonla ve tum bagimliliklari kur
!git clone https://github.com/hkchengrex/MMAudio.git
%cd MMAudio
!pip install -e . -q

print("\nKurulum tamamlandi!")

In [ ]:
# Model agirliklarini indir (ilk seferde ~2-3 dakika surer)
print("Model indiriliyor, lutfen bekleyin...")

from mmaudio.eval_utils import all_model_cfg

# Secilen modeli yukle
model_cfg = all_model_cfg[MODEL_NAME]

# Model dosyalari yoksa otomatik indirir
model_cfg.download_if_needed()

print(f"Model basariyla indirildi: {MODEL_NAME}")

In [ ]:
# Ses uretimini baslat
import torch
import torchaudio
from mmaudio.eval_utils import setup_eval_logging, generate
from mmaudio.model.networks import get_my_mmaudio
from mmaudio.model.flow_matching import FlowMatching
from mmaudio.model.utils.features_utils import FeaturesUtils

# TF32 hizlandirmayi aktif et
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

setup_eval_logging()

# Cihaz ve veri tipini ayarla (T4 icin float16 kullaniyoruz)
device = 'cuda'
dtype = torch.float16  # T4 GPU bfloat16 desteklemez

# Modeli yukle
print("Model yukleniyor...")
net = get_my_mmaudio(model_cfg.model_name).to(device, dtype).eval()
net.load_weights(torch.load(model_cfg.model_path, map_location=device, weights_only=True))

# Rastgele sayi ureteci
rng = torch.Generator(device=device)
rng.manual_seed(SEED)

# Flow matching ayari
fm = FlowMatching(min_sigma=0, inference_mode=INFERENCE_MODE, num_steps=NUM_STEPS)

# Ozellik cikarici ayarla
feature_utils = FeaturesUtils(
    tod_vae_ckpt=model_cfg.vae_path,
    synchformer_ckpt=model_cfg.synchformer_ckpt,
    enable_conditions=True,
    mode=model_cfg.mode,
    bigvgan_vocoder_ckpt=model_cfg.bigvgan_16k_path,
    need_vae_encoder=False
)
feature_utils = feature_utils.to(device, dtype).eval()

# Video yukle (eger verilmisse)
clip_video = None
sync_video = None
if VIDEO_PATH is not None:
    from mmaudio.utils.video import load_video
    print(f"Video yukleniyor: {VIDEO_PATH}")
    clip_video, sync_video, duration_sec = load_video(VIDEO_PATH, DURATION)
    clip_video = clip_video.unsqueeze(0).to(device, dtype)
    sync_video = sync_video.unsqueeze(0).to(device, dtype)
    print(f"Video yuklendi ({duration_sec:.1f}s)")

# Ses suresini ayarla
seq_cfg = model_cfg.seq_cfg
seq_cfg.duration = DURATION
net.update_seq_lengths(seq_cfg.latent_seq_len, seq_cfg.clip_seq_len, seq_cfg.sync_seq_len)

# Ses uret
mod = "video-to-audio" if VIDEO_PATH else "text-to-audio"
print(f"Ses uretiliyor ({mod}): '{PROMPT}'...")
with torch.inference_mode():
    audios = generate(
        clip_video=clip_video,
        sync_video=sync_video,
        text=[PROMPT],
        negative_text=[NEGATIVE_PROMPT],
        feature_utils=feature_utils,
        net=net,
        fm=fm,
        rng=rng,
        cfg_strength=CFG_STRENGTH,
    )

# Sonucu kaydet
audio = audios.float().cpu()[0]
output_file = 'result.flac'
torchaudio.save(output_file, audio, seq_cfg.sampling_rate)

print(f"\nSes basariyla uretildi ve '{output_file}' olarak kaydedildi!")
print(f"Ornekleme hizi: {seq_cfg.sampling_rate} Hz")
print(f"Sure: {DURATION} saniye")

In [ ]:
# Dosyayi bilgisayariniza indirin
from google.colab import files

print("result.flac indiriliyor...")

# Dosya var mi kontrol et
import os
if os.path.exists('result.flac'):
    # Dosyayi indir
    files.download('result.flac')
    print("Indirme baslatildi! Tarayicinizin indirilenler klasorune bakin.")
else:
    print("HATA: result.flac bulunamadi!")
    print("Lutfen onceki hucreleri sirayla calistirdiginizdan emin olun.")